# Diagnostic H0 Tabular Sweep - Validation Only Diagnostic

Protocol: `docs/DIAGNOSTIC_H0_TABULAR_SWEEP_PROTOCOL_2026-06-04.md`

Scope: `validation_only_diagnostic`

This notebook is a post-Stage 0 diagnostic. It does not replace official Stage
0A2/0B outputs, does not authorize holdout/test use, and must not be used as a
thesis result claim.

Entry conditions:

1. `02_config_screening_colab.ipynb` has completed Stage 0A2.
2. Stage 0B has completed, or its blocker has been explicitly recorded.
3. `/content/stage0_config_screening_results/` contains Stage 0 output files.
4. Holdout/test remains closed.

If those Stage 0 files are unavailable in a fresh Colab runtime, the notebook
will first search for an extracted Stage 0 artifact folder. If neither official
outputs nor extracted artifacts are available, it can still run in
`part0_standalone` mode. In that mode Part 0 defines the baseline for this
diagnostic run, but the output cannot claim official Stage 0A2 reproduction.

This notebook is configured for one-pass Colab execution after Stage 0B output
files exist: `RUN_H0_FULL_SEQUENCE = True` runs Part 0, Part 1, Part 2, and
Part 3 in order. Google Drive backup is enabled by default and copies outputs
after each completed part.


In [ ]:
from pathlib import Path
from collections import Counter
import importlib
import json
import math
import random
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 160)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

INSTALL_LIGHTGBM_IF_MISSING = True
INSTALL_TORCH_IF_MISSING = False
PYTHON_DEPS_DIR = Path("/content/stage0_python_deps")


def install_package_to_local_target(package_name):
    target_dir = PYTHON_DEPS_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    base_name = package_name.split("[", 1)[0].replace("-", "_")
    for pattern in (base_name, f"{base_name}-*.dist-info"):
        for path in target_dir.glob(pattern):
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--target",
        str(target_dir),
        package_name,
    ]
    print("Running dependency install:", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if proc.returncode != 0:
        raise RuntimeError(
            f"pip install failed for {package_name} with exit code {proc.returncode}. "
            "Read the pip output above. If Colab reports a package or filesystem "
            "error, restart the runtime and rerun setup cells."
        )
    target_text = str(target_dir)
    if target_text not in sys.path:
        sys.path.insert(0, target_text)
    importlib.invalidate_caches()


def ensure_lightgbm():
    try:
        import lightgbm as lgb
        return lgb
    except (ImportError, OSError) as exc:
        if INSTALL_LIGHTGBM_IF_MISSING:
            print(f"LightGBM import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("lightgbm")
            sys.modules.pop("lightgbm", None)
            try:
                import lightgbm as lgb
                return lgb
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "LightGBM still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0S."
                ) from retry_exc
        raise RuntimeError(
            "LightGBM import failed. Set INSTALL_LIGHTGBM_IF_MISSING=True and "
            "rerun setup cells. The notebook will install LightGBM into "
            f"{PYTHON_DEPS_DIR} and place that directory first on sys.path."
        ) from exc


def ensure_torch():
    try:
        import torch
        return torch
    except (ImportError, OSError) as exc:
        if INSTALL_TORCH_IF_MISSING:
            print(f"PyTorch import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("torch")
            sys.modules.pop("torch", None)
            try:
                import torch
                return torch
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "PyTorch still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0B."
                ) from retry_exc
        raise RuntimeError(
            "PyTorch import failed. Set INSTALL_TORCH_IF_MISSING=True and rerun "
            "setup cells before enabling Stage 0B deep models."
        ) from exc


In [ ]:
\
TICKERS = ("CSCO", "JPM", "KO", "MSFT", "WMT")
MODEL_SEEDS = (101, 202, 303, 404, 505)
H0_FRESH_SEEDS = (606, 707, 808, 909, 1010)

RUN_H0_FULL_SEQUENCE = True
RUN_H0_PART0 = RUN_H0_FULL_SEQUENCE
RUN_H0_PART1 = RUN_H0_FULL_SEQUENCE
RUN_H0_PART2 = RUN_H0_FULL_SEQUENCE
RUN_H0_PART3 = RUN_H0_FULL_SEQUENCE
RUN_H0_ROUND2_IF_TRIGGERED = True

BACKUP_TO_GOOGLE_DRIVE = True
BACKUP_FAILURE_IS_FATAL = True
ALLOW_STANDALONE_H0_BASELINE = True
DRIVE_BACKUP_DIR = Path(
    "/content/drive/MyDrive/intraday_stock_direction_research/diagnostic_h0_tabular_sweep"
)

DIAGNOSTIC_NAME = "diagnostic_h0_tabular_sweep"
RESULT_SCOPE = "validation_only_diagnostic"

H0_LABEL_CONFIG = "h03_bps1p5"
H0_FEATURE_SET = "price_volume_time"
H0_BASELINE_MODEL = "lightgbm"
H0_BASELINE_PROFILE = "profile_B"
H0_BASELINE_WINDOW = 20
H0_BASELINE_TOLERANCE = 1e-4

H0_WEAK_DELTA = 0.005
H0_STRONG_DELTA = 0.010
H0_MIN_POSITIVE_TICKERS = 4
H0_MAX_TOP_TICKER_GAIN_SHARE = 0.50
H0_LOGREG_WARNING_RATE_ABORT = 0.05

H0_PART1_LGBM_WINDOWS = (6, 12, 16, 20, 24, 28, 32, 48, 64)
H0_PART1_LOGREG_WINDOWS = (12, 20, 24, 32, 48)
H0_ROUND2_TRIGGER_WINDOWS = (24, 28, 32)
H0_ROUND2_WINDOWS = (18, 20, 22, 24, 26, 28, 30, 32, 36)
H0_PART2_WINDOWS = (20, 24, 32)

ALL_FEATURES = (
    "log_return",
    "close_to_open_return",
    "high_low_range",
    "rolling_volatility_20",
    "normalized_volume_20",
    "rsi_14",
    "bollinger_pctb",
    "normalized_macd_hist",
    "time_of_day_sin",
    "time_of_day_cos",
)

FEATURE_SETS = {
    "price_action_core": (
        "log_return",
        "close_to_open_return",
        "high_low_range",
    ),
    "technical_price": (
        "log_return",
        "high_low_range",
        "rsi_14",
        "bollinger_pctb",
        "normalized_macd_hist",
    ),
    "price_volume_time": ALL_FEATURES,
}

LABEL_CONFIGS = {
    "h03_bps1p5": {"horizon_k": 3, "threshold_bps": 1.5},
    "h09_bps3p0": {"horizon_k": 9, "threshold_bps": 3.0},
    "h24_bps7p5": {"horizon_k": 24, "threshold_bps": 7.5},
}

MAX_TRAIN_ROWS = None
RANDOM_SUBSAMPLE_SEED = 42

LGBM_PROFILES = {
    "profile_A": {
        "n_estimators": 150,
        "learning_rate": 0.05,
        "max_depth": 3,
        "num_leaves": 7,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
    },
    "profile_B": {
        "n_estimators": 200,
        "learning_rate": 0.03,
        "max_depth": 6,
        "num_leaves": 31,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
    },
    "profile_C": {
        "n_estimators": 300,
        "learning_rate": 0.02,
        "max_depth": 8,
        "num_leaves": 63,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
    },
    "profile_D1": {
        "n_estimators": 200,
        "learning_rate": 0.03,
        "max_depth": 6,
        "num_leaves": 31,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
        "min_child_samples": 100,
    },
    "profile_D2": {
        "n_estimators": 200,
        "learning_rate": 0.03,
        "max_depth": 6,
        "num_leaves": 31,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
    },
    "profile_D3": {
        "n_estimators": 200,
        "learning_rate": 0.03,
        "max_depth": 6,
        "num_leaves": 31,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
        "min_child_samples": 100,
        "reg_lambda": 1.0,
    },
}

LGBM_PARAMS = LGBM_PROFILES[H0_BASELINE_PROFILE].copy()
LOGREG_PROFILES = {
    "logreg_l2_c1": {"penalty": "l2", "C": 1.0, "solver": "liblinear"},
}

TRAIN_START, TRAIN_END = "1998-01-02", "2013-09-16"
VAL_START, VAL_END = "2013-09-16", "2017-01-25"
CALENDAR_SPLITS = {
    "train": (TRAIN_START, TRAIN_END),
    "validation": (VAL_START, VAL_END),
}

BPS_TO_DECIMAL = 10000.0
BAR_INTERVAL_MINUTES = 5
MARKET_OPEN_MINUTE = 9 * 60 + 30
TRADING_DAY_MINUTES = 390
TIME_OF_DAY_ENCODING_PERIOD_MINUTES = TRADING_DAY_MINUTES + BAR_INTERVAL_MINUTES
MARKET_OPEN = pd.Timestamp("09:30").time()
MARKET_CLOSE = pd.Timestamp("16:00").time()
EXPECTED_COLUMNS = ("timestamp", "open", "high", "low", "close", "volume")
DATA_FILE_SUFFIXES = (".csv", ".txt")
RAW_TXT_COLUMNS = ("Date", "Time", "Open", "High", "Low", "Close", "Volume")

STAGE0_OUTPUT_DIR = Path("/content/stage0_config_screening_results")
STAGE0_FILES = {
    "stage0a2_pooled": STAGE0_OUTPUT_DIR / "stage0a2_pooled.csv",
    "stage0a2_summary": STAGE0_OUTPUT_DIR / "stage0a2_summary.csv",
    "stage0_candidates": STAGE0_OUTPUT_DIR / "stage0_candidates.json",
    "stage0b_summary": STAGE0_OUTPUT_DIR / "stage0b_summary.csv",
}
STAGE0_ARTIFACT_DIR_CANDIDATES = (
    Path("artifacts/stage0_desktop_02_config_screening_2026-06-04"),
    Path("/content/artifacts/stage0_desktop_02_config_screening_2026-06-04"),
    Path("/content/stage0_desktop_02_config_screening_2026-06-04"),
    Path("/content/drive/MyDrive/intraday_stock_direction_research/artifacts/stage0_desktop_02_config_screening_2026-06-04"),
    Path("/content/drive/MyDrive/intraday_stock_direction_research/stage0_desktop_02_config_screening_2026-06-04"),
    Path("/content/drive/MyDrive/stage0_desktop_02_config_screening_2026-06-04"),
)
STAGE0_ARTIFACT_FILES = {
    "stage0a2_summary": "stage0a2_table1.csv",
    "stage0_candidates": "stage0_decision_blocks.json",
    "stage0b_summary": "stage0b_table1.csv",
}

OUTPUT_DIR = Path("/content/diagnostic_h0_tabular_sweep")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILES = {
    "rules": OUTPUT_DIR / "diagnostic_h0_pre_committed_rules.json",
    "part0": OUTPUT_DIR / "diagnostic_h0_part0_sanity.csv",
    "part1": OUTPUT_DIR / "diagnostic_h0_part1_window_sweep.csv",
    "part2": OUTPUT_DIR / "diagnostic_h0_part2_lgbm_profiles.csv",
    "part3": OUTPUT_DIR / "diagnostic_h0_part3_confirmation.csv",
    "summary": OUTPUT_DIR / "diagnostic_h0_summary.csv",
    "per_ticker": OUTPUT_DIR / "diagnostic_h0_per_ticker.csv",
}

PRECOMMITTED_RULES = {
    "diagnostic_name": DIAGNOSTIC_NAME,
    "scope": RESULT_SCOPE,
    "baseline_source": str(STAGE0_FILES["stage0a2_summary"]),
    "candidate_source": str(STAGE0_FILES["stage0_candidates"]),
    "artifact_dir_candidates": [str(path) for path in STAGE0_ARTIFACT_DIR_CANDIDATES],
    "allow_standalone_h0_baseline": ALLOW_STANDALONE_H0_BASELINE,
    "baseline_cell": {
        "model": H0_BASELINE_MODEL,
        "label_config": H0_LABEL_CONFIG,
        "feature_set": H0_FEATURE_SET,
        "window_size": H0_BASELINE_WINDOW,
        "seeds": MODEL_SEEDS,
    },
    "part0_tolerance": H0_BASELINE_TOLERANCE,
    "weak_delta": H0_WEAK_DELTA,
    "strong_delta": H0_STRONG_DELTA,
    "positive_ticker_count_min": H0_MIN_POSITIVE_TICKERS,
    "top_ticker_gain_share_max": H0_MAX_TOP_TICKER_GAIN_SHARE,
    "round2_trigger_windows": H0_ROUND2_TRIGGER_WINDOWS,
    "round2_windows": H0_ROUND2_WINDOWS,
    "fresh_seeds": H0_FRESH_SEEDS,
}

with OUTPUT_FILES["rules"].open("w", encoding="utf-8") as handle:
    json.dump(PRECOMMITTED_RULES, handle, indent=2)

display(pd.DataFrame([
    {"feature_set": name, "n_features": len(features), "features": ", ".join(features)}
    for name, features in FEATURE_SETS.items()
]))
print("Stage 0 output directory:", STAGE0_OUTPUT_DIR)
print("Stage 0 artifact candidates:", [str(path) for path in STAGE0_ARTIFACT_DIR_CANDIDATES])
print("Diagnostic H0 output directory:", OUTPUT_DIR)
print("Drive backup enabled:", BACKUP_TO_GOOGLE_DRIVE)
print("Drive backup directory:", DRIVE_BACKUP_DIR)
print("Standalone H0 baseline fallback enabled:", ALLOW_STANDALONE_H0_BASELINE)
print("Run switches:", {
    "RUN_H0_FULL_SEQUENCE": RUN_H0_FULL_SEQUENCE,
    "RUN_H0_PART0": RUN_H0_PART0,
    "RUN_H0_PART1": RUN_H0_PART1,
    "RUN_H0_PART2": RUN_H0_PART2,
    "RUN_H0_PART3": RUN_H0_PART3,
    "RUN_H0_ROUND2_IF_TRIGGERED": RUN_H0_ROUND2_IF_TRIGGERED,
})


## Data Loading

This cell resolves the five real ticker files and downloads them through the
Google Drive API when needed. It does not mount MyDrive. For raw `.txt` files it
first scans to the validation boundary and then reads only rows before
`VAL_END`, so closed holdout/test rows are not materialized into the notebook
dataframes. Outputs stay local under `/content/stage0_config_screening_results`.


In [ ]:
RAW_DRIVE_FOLDER_ID = "154SlcH3nViUcvPXFBM-E4NPg_ybljBTG"
RAW_DRIVE_FOLDER_NAME = "s&p 100 adjusted 1 min data"
RAW_DATA_DIR = Path("/content/stage0_raw_stock_data")
DOWNLOAD_RAW_DATA_FROM_DRIVE = True

RAW_DRIVE_FILES = {
    "CSCO": {"name": "CSCO.txt", "file_id": "17A49kUiMELuQqdkOhw1KrpudjP5i5xIN"},
    "JPM": {"name": "JPM.txt", "file_id": "11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q"},
    "KO": {"name": "KO.txt", "file_id": "1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn"},
    "MSFT": {"name": "MSFT.txt", "file_id": "1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs"},
    "WMT": {"name": "WMT.txt", "file_id": "1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c"},
}


def is_real_raw_file(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.name.lower().endswith(".gshortcut"):
        return False
    if path.suffix.lower() not in DATA_FILE_SUFFIXES:
        return False
    try:
        return path.stat().st_size > 50_000
    except OSError:
        return False


def build_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Drive API is unavailable. Open this notebook in Google Colab and "
            "authenticate when prompted; do not use drive.mount for Stage 0 data."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")


def download_raw_drive_files():
    if not DOWNLOAD_RAW_DATA_FROM_DRIVE:
        return
    try:
        from googleapiclient.http import MediaIoBaseDownload
    except ImportError as exc:
        raise RuntimeError("googleapiclient is unavailable in this Colab runtime.") from exc

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    service = build_drive_service()
    for ticker, item in RAW_DRIVE_FILES.items():
        target = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(target):
            print(f"{ticker}: using cached raw file {target}")
            continue
        print(f"{ticker}: downloading raw Drive file {item['name']} -> {target}")
        request = service.files().get_media(fileId=item["file_id"])
        with target.open("wb") as output:
            downloader = MediaIoBaseDownload(output, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        if not is_real_raw_file(target):
            raise ValueError(f"Downloaded file is not a real raw ticker file: {target}")


def resolve_data_files():
    if DOWNLOAD_RAW_DATA_FROM_DRIVE:
        download_raw_drive_files()
    files = {}
    missing = []
    for ticker, item in RAW_DRIVE_FILES.items():
        path = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(path):
            files[ticker] = path
        else:
            missing.append(f"{ticker}: {path}")
    if missing:
        raise FileNotFoundError(
            "Missing required raw ticker files after Drive API resolution: "
            + "; ".join(missing)
        )
    print("resolved raw Drive data files:")
    for ticker, path in files.items():
        print(f"  {ticker}: {path}")
    return files


def find_timestamp_column(columns):
    for candidate in ("timestamp", "datetime", "date", "time"):
        for column in columns:
            if str(column).lower() == candidate:
                return column
    raise ValueError(f"No timestamp-like column found in columns: {list(columns)}")


def normalize_ohlcv_columns(frame, source_name):
    lower_map = {str(column).lower(): column for column in frame.columns}
    rename = {}
    for required in EXPECTED_COLUMNS:
        if required == "timestamp":
            continue
        if required not in lower_map:
            raise ValueError(f"{source_name} missing required column: {required}")
        rename[lower_map[required]] = required
    return frame.rename(columns=rename)


def resample_to_five_minutes(frame):
    resampled = (
        frame.set_index("timestamp")
        .resample("5min")
        .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
        .dropna(subset=["open", "high", "low", "close", "volume"])
        .reset_index()
    )
    times = resampled["timestamp"].dt.time
    return resampled.loc[
        (times >= MARKET_OPEN) & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ].reset_index(drop=True)


def txt_date_key(date_text):
    parts = str(date_text).strip().split("/")
    if len(parts) != 3:
        raise ValueError(f"Unexpected Date field in raw txt file: {date_text!r}")
    month, day, year = parts
    return int(year), int(month), int(day)


def count_txt_rows_before_validation_end(path):
    validation_end_key = txt_date_key(pd.Timestamp(VAL_END).strftime("%m/%d/%Y"))
    safe_rows = 0
    has_header = False
    reached_boundary = False
    with Path(path).open("rt", encoding="utf-8", errors="replace", newline="") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped:
                continue
            first_field = stripped.split(",", 1)[0].strip()
            if first_field.lower() == "date":
                has_header = True
                continue
            if txt_date_key(first_field) >= validation_end_key:
                reached_boundary = True
                break
            safe_rows += 1
    if safe_rows == 0:
        raise ValueError(f"No train/validation rows found before {VAL_END} in: {path}")
    if not reached_boundary:
        print(f"{Path(path).name}: no row at or after {VAL_END} found; read capped to file end.")
    return safe_rows, has_header


def read_one_minute_txt(path):
    safe_rows, has_header = count_txt_rows_before_validation_end(path)
    print(f"{Path(path).name}: loading {safe_rows:,} raw one-minute rows before {VAL_END}.")
    frame = pd.read_csv(
        path,
        header=None,
        names=RAW_TXT_COLUMNS,
        nrows=safe_rows,
        skiprows=1 if has_header else None,
        low_memory=False,
    )
    frame = frame.loc[frame["Date"].astype(str).str.lower() != "date"].reset_index(drop=True)
    frame["timestamp"] = pd.to_datetime(
        frame["Date"].astype(str) + " " + frame["Time"].astype(str),
        format="%m/%d/%Y %H:%M",
        errors="raise",
    )
    frame = frame.drop(columns=["Date", "Time"]).rename(
        columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    )
    numeric_columns = ["open", "high", "low", "close", "volume"]
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors="raise")
    validation_end = pd.Timestamp(VAL_END)
    times = frame["timestamp"].dt.time
    frame = frame.loc[
        (frame["timestamp"] < validation_end)
        & (times >= MARKET_OPEN)
        & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ]
    return resample_to_five_minutes(frame)


def read_five_minute_csv(path):
    validation_end = pd.Timestamp(VAL_END)
    chunks = []
    for chunk in pd.read_csv(path, chunksize=100_000):
        timestamp_column = find_timestamp_column(chunk.columns)
        chunk = chunk.rename(columns={timestamp_column: "timestamp"}).copy()
        chunk = normalize_ohlcv_columns(chunk, path.name)
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], errors="raise")
        raw_chunk_max_timestamp = chunk["timestamp"].max()
        chunk = chunk.loc[chunk["timestamp"] < validation_end, list(EXPECTED_COLUMNS)]
        if not chunk.empty:
            chunks.append(chunk)
        if raw_chunk_max_timestamp >= validation_end:
            break
    if not chunks:
        raise ValueError(f"No train/validation rows found in: {path}")
    return pd.concat(chunks, ignore_index=True)


def load_ticker(ticker, path):
    path = Path(path)
    frame = read_one_minute_txt(path) if path.suffix.lower() == ".txt" else read_five_minute_csv(path)
    frame["ticker"] = ticker
    return frame.sort_values("timestamp").reset_index(drop=True)


RUN_ANY_STAGE = bool(RUN_H0_PART0 or RUN_H0_PART1 or RUN_H0_PART2 or RUN_H0_PART3)

if RUN_ANY_STAGE:
    DATA_FILES = resolve_data_files()
    raw_data = {ticker: load_ticker(ticker, DATA_FILES[ticker]) for ticker in TICKERS}

    display(pd.DataFrame([
        {
            "ticker": ticker,
            "rows": len(frame),
            "start": frame["timestamp"].min(),
            "end": frame["timestamp"].max(),
            "source": DATA_FILES[ticker].name,
            "path": str(DATA_FILES[ticker]),
        }
        for ticker, frame in raw_data.items()
    ]))
else:
    DATA_FILES = {}
    raw_data = {}
    print("All RUN_H0* switches are False; data loading skipped.")


## Feature, Label, Split, Scale, Window

These functions implement the active chronology-safe Stage 0 contracts inside a
standalone Colab notebook. They do not import a local helper package or a prior
route. Keep the causal post-bar-close feature rules, cumulative
horizon-return labels, split-boundary invalidation, train-only scaler fitting,
and per-ticker/per-day windows aligned with the active freeze documents before
rerunning Stage 0.

Tabular models use flattened windows: each LogReg/LightGBM sample is
`window_size * n_features`, built from the same per-ticker/per-day rows used by
the sequence models. This makes Stage 0A2 a true window-length sensitivity
check rather than a last-bar sample-count check.

Feature timing boundary: prediction is after the current five-minute bar has
completed, so `close[t]`, `high[t]`, `low[t]`, `volume[t]`, and same-row
timestamp encodings are available. Same-day trailing Bollinger windows may
include `close[t]`; RSI and MACD EWM states are causal but intentionally
continuous across trading days.


In [ ]:
def finite_mask(frame, columns):
    return np.isfinite(frame[list(columns)].to_numpy(dtype=float)).all(axis=1)


def grouped_rolling(series, group_key, window, how):
    grouped = series.groupby(group_key, group_keys=False)
    if how == "mean":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).mean())
    if how == "std":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).std())
    raise ValueError(f"Unknown rolling operation: {how}")


def continuous_ewm(series, span):
    return series.ewm(span=span, adjust=False, min_periods=span).mean()


def continuous_wilder_ewm(series, period):
    return series.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()


def validated_ohlcv(frame):
    out = frame[["open", "high", "low", "close", "volume"]].astype(float)
    bad = (
        (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"] < 0)
        | (out["high"] < out["low"])
        | (out["open"] < out["low"])
        | (out["open"] > out["high"])
        | (out["close"] < out["low"])
        | (out["close"] > out["high"])
    )
    if bad.any():
        raise ValueError(f"Invalid OHLCV rows found: {int(bad.sum())}")
    return out


def add_features(frame):
    current = frame.sort_values("timestamp").copy()
    day = current["timestamp"].dt.date
    ohlcv = validated_ohlcv(current)
    close, open_, high, low, volume = (ohlcv[c] for c in ["close", "open", "high", "low", "volume"])

    log_close = np.log(close)
    current["log_return"] = log_close.groupby(day, group_keys=False).diff()
    current["close_to_open_return"] = close / open_ - 1.0
    current["high_low_range"] = np.log(high / low)

    shifted_log_return = current["log_return"].groupby(day, group_keys=False).shift(1)
    current["rolling_volatility_20"] = grouped_rolling(shifted_log_return, day, 20, "std")

    log_volume = np.log1p(volume)
    shifted_log_volume = log_volume.groupby(day, group_keys=False).shift(1)
    current["normalized_volume_20"] = log_volume - grouped_rolling(shifted_log_volume, day, 20, "mean")

    close_delta = close.groupby(day, group_keys=False).diff()
    gain = close_delta.clip(lower=0.0)
    loss = (-close_delta).clip(lower=0.0)
    avg_gain = continuous_wilder_ewm(gain, 14)
    avg_loss = continuous_wilder_ewm(loss, 14)
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask(avg_loss.eq(0.0) & avg_gain.gt(0.0), 100.0)
    current["rsi_14"] = rsi.mask(avg_loss.eq(0.0) & avg_gain.eq(0.0), 50.0)

    rolling_mean_20 = grouped_rolling(close, day, 20, "mean")
    rolling_std_20 = grouped_rolling(close, day, 20, "std")
    lower_band = rolling_mean_20 - 2.0 * rolling_std_20
    upper_band = rolling_mean_20 + 2.0 * rolling_std_20
    current["bollinger_pctb"] = (close - lower_band) / (upper_band - lower_band).replace(0.0, np.nan)

    ema_12 = continuous_ewm(close, 12)
    ema_26 = continuous_ewm(close, 26)
    macd = ema_12 - ema_26
    signal = continuous_ewm(macd, 9)
    current["normalized_macd_hist"] = (macd - signal) / ema_26.replace(0.0, np.nan)

    minute_of_day = current["timestamp"].dt.hour * 60 + current["timestamp"].dt.minute
    minutes_since_open = minute_of_day - MARKET_OPEN_MINUTE
    current["time_of_day_sin"] = np.sin(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    current["time_of_day_cos"] = np.cos(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    return current.replace([np.inf, -np.inf], np.nan)


def add_labels(frame, horizon_k, threshold_bps):
    current = frame.sort_values("timestamp").copy()
    threshold = threshold_bps / BPS_TO_DECIMAL
    close = current["close"].astype(float)
    future_timestamp = current["timestamp"].shift(-horizon_k)
    current["future_cumulative_return"] = close.shift(-horizon_k) / close - 1.0

    same_day = pd.Series(True, index=current.index)
    current_day = current["timestamp"].dt.date
    for offset in range(1, horizon_k + 1):
        same_day &= current_day.shift(-offset).eq(current_day)

    actual_horizon = future_timestamp - current["timestamp"]
    expected_horizon = pd.Timedelta(minutes=BAR_INTERVAL_MINUTES * horizon_k)
    current["diagnostic_irregular_horizon"] = (
        future_timestamp.notna() & same_day & actual_horizon.ne(expected_horizon)
    )
    current["invalid_irregular_horizon"] = current["diagnostic_irregular_horizon"]
    current["invalid_missing_future"] = current["future_cumulative_return"].isna()
    current["invalid_cross_day"] = ~same_day
    invalid = (
        current["invalid_missing_future"]
        | current["invalid_cross_day"]
        | current["invalid_irregular_horizon"]
    )
    current["label"] = np.nan
    valid = ~invalid
    current.loc[invalid, "future_cumulative_return"] = np.nan
    current.loc[valid & (current["future_cumulative_return"] > threshold), "label"] = 1
    current.loc[valid & (current["future_cumulative_return"] < -threshold), "label"] = 0
    return current


def assign_split(timestamp):
    ts = pd.Timestamp(timestamp)
    for split_name, (start, end) in CALENDAR_SPLITS.items():
        if pd.Timestamp(start) <= ts < pd.Timestamp(end):
            return split_name
    return "outside_train_validation"


def add_split_and_invalidate_boundaries(frame, horizon_k):
    current = frame.sort_values("timestamp").copy()
    current["split"] = current["timestamp"].map(assign_split)
    horizon_split = current["split"].shift(-horizon_k)
    cross_split = current["future_cumulative_return"].notna() & (current["split"] != horizon_split)
    current["invalid_cross_split"] = cross_split
    current.loc[cross_split, "label"] = np.nan
    current.loc[cross_split, "future_cumulative_return"] = np.nan
    return current


def prepare_split_frames(raw_frames, horizon_k, threshold_bps):
    split_frames = {}
    for ticker, frame in raw_frames.items():
        featured = add_features(frame)
        labeled = add_labels(featured, horizon_k=horizon_k, threshold_bps=threshold_bps)
        split_frames[ticker] = add_split_and_invalidate_boundaries(labeled, horizon_k=horizon_k)
    return split_frames


def fit_transform_train_validation(split_frames, feature_columns):
    train_parts = []
    for frame in split_frames.values():
        train = frame.loc[frame["split"] == "train", list(feature_columns)]
        train_parts.append(train.loc[finite_mask(train, feature_columns)])
    train_matrix = pd.concat(train_parts, axis=0)
    if train_matrix.empty:
        raise ValueError("No train rows available for scaler fit.")

    scaler = StandardScaler().fit(train_matrix)
    scaled = {}
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    for ticker, frame in split_frames.items():
        current = frame.copy()
        for col in scaled_columns:
            current[col] = np.nan
        eligible = current["split"].isin(["train", "validation"])
        complete = finite_mask(current, feature_columns)
        rows = eligible & complete
        if rows.any():
            current.loc[rows, scaled_columns] = scaler.transform(current.loc[rows, feature_columns])
        scaled[ticker] = current
    return scaled


def build_last_step_windows(frames_by_ticker, feature_columns, split_name, window_size):
    # Flatten the past `window_size` bars into a (window_size * n_features,) vector per sample.
    # Tabular models (LogReg/LightGBM) thus consume lagged history instead of only the last bar,
    # which makes window_size a meaningful search dimension for them too.
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    n_features = len(feature_columns)
    flat_dim = window_size * n_features
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1].reshape(-1))
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])

    if not x_parts:
        return (
            np.empty((0, flat_dim), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


## Diagnostic H0 Model And Metrics Helpers

This section reuses Stage 0 helper definitions, then H0 overrides the tabular run layer below.

In [ ]:
DATASET_CACHE = {}


def label_spec(label_config):
    spec = LABEL_CONFIGS[label_config]
    return int(spec["horizon_k"]), float(spec["threshold_bps"])


def subsample_rows_uniformly(x_values, y_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected]


def subsample_rows_with_owner(x_values, y_values, owner_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values, owner_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected], owner_values[selected]


def evaluate_predictions(y_true, predictions):
    return {
        "macro_f1": float(f1_score(y_true, predictions, labels=[0, 1], average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "accuracy": float(accuracy_score(y_true, predictions)),
    }


def dummy_metrics(y_train, y_validation, seed):
    if len(y_train) == 0 or len(y_validation) == 0:
        return {"dummy_macro_f1": np.nan, "dummy_balanced_accuracy": np.nan}
    x_train = np.zeros((len(y_train), 1))
    x_validation = np.zeros((len(y_validation), 1))
    dummy = DummyClassifier(strategy="stratified", random_state=seed).fit(x_train, y_train)
    pred = dummy.predict(x_validation)
    return {
        "dummy_macro_f1": float(f1_score(y_validation, pred, labels=[0, 1], average="macro", zero_division=0)),
        "dummy_balanced_accuracy": float(balanced_accuracy_score(y_validation, pred)),
    }


def sample_std(values):
    values = pd.Series(values).dropna()
    return float(values.std(ddof=1)) if len(values) > 1 else 0.0


T_CRITICAL_ONE_SIDED_95 = {
    2: 6.314,
    3: 2.920,
    4: 2.353,
    5: 2.132,
    6: 2.015,
    7: 1.943,
    8: 1.895,
    9: 1.860,
    10: 1.833,
    11: 1.812,
    12: 1.796,
}


def t_critical_one_sided_95(seed_count):
    if seed_count <= 1:
        return 0.0
    return T_CRITICAL_ONE_SIDED_95.get(int(seed_count), 1.645)


def build_sequence_windows(frames_by_ticker, feature_columns, split_name, window_size):
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1])
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])
    if not x_parts:
        return (
            np.empty((0, window_size, len(feature_columns)), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts).astype(np.float32),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


def get_dataset(label_config, feature_set, window_size):
    key = (label_config, feature_set, int(window_size))
    if key in DATASET_CACHE:
        dataset = DATASET_CACHE[key].copy()
        dataset["prep_seconds"] = 0.0
        return dataset
    if not raw_data:
        raise RuntimeError("raw_data is empty. Enable a RUN_STAGE0* switch and rerun data loading first.")
    horizon_k, threshold_bps = label_spec(label_config)
    feature_columns = FEATURE_SETS[feature_set]
    start = time.perf_counter()
    split_frames = prepare_split_frames(raw_data, horizon_k=horizon_k, threshold_bps=threshold_bps)
    scaled_frames = fit_transform_train_validation(split_frames, feature_columns)
    x_train, y_train, train_owner, train_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation, y_validation, validation_owner, validation_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    x_train_seq, y_train_seq, train_owner_seq, train_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation_seq, y_validation_seq, validation_owner_seq, validation_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    if len(y_train) == 0 or len(y_validation) == 0:
        raise ValueError(f"No tabular windows available for {label_config} / {feature_set} / window={window_size}")
    if len(y_train_seq) != len(y_train) or len(y_validation_seq) != len(y_validation):
        raise ValueError("Tabular and sequence window counts disagree; inspect window construction.")
    dataset = {
        "label_config": label_config,
        "horizon_k": horizon_k,
        "threshold_bps": threshold_bps,
        "feature_set": feature_set,
        "feature_columns": feature_columns,
        "window_size": int(window_size),
        "x_train": x_train,
        "y_train": y_train,
        "train_owner": train_owner,
        "x_validation": x_validation,
        "y_validation": y_validation,
        "validation_owner": validation_owner,
        "x_train_seq": x_train_seq,
        "y_train_seq": y_train_seq,
        "train_owner_seq": train_owner_seq,
        "x_validation_seq": x_validation_seq,
        "y_validation_seq": y_validation_seq,
        "validation_owner_seq": validation_owner_seq,
        "prep_seconds": time.perf_counter() - start,
    }
    DATASET_CACHE[key] = dataset.copy()
    return dataset


def fit_predict_logreg(dataset, seed):
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    # Tabular features are flattened windows, so allow more iterations for the
    # higher-dimensional design matrix.
    max_iter = 2000
    model = LogisticRegression(
        solver="liblinear",
        max_iter=max_iter,
        class_weight="balanced",
        random_state=seed,
    )
    start_fit = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    max_iter_reached = bool((model.n_iter_ >= max_iter).any())
    fit_status = "converged" if not convergence_warnings and not max_iter_reached else "convergence_warning"
    return pred, fit_seconds, predict_seconds, int(len(y_train)), fit_status


def fit_predict_lightgbm(dataset, seed):
    lgb = ensure_lightgbm()
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    model = lgb.LGBMClassifier(
        **LGBM_PARAMS,
        class_weight="balanced",
        random_state=seed,
        verbosity=-1,
    )
    start_fit = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    return pred, fit_seconds, predict_seconds, int(len(y_train)), "not_applicable"


def set_global_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch = ensure_torch()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return torch


def make_simple_gru(input_dim, seed):
    torch = set_global_seed(seed)
    nn = torch.nn

    class SimpleGRUClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.gru = nn.GRU(input_dim, 32, num_layers=1, batch_first=True)
            self.dropout = nn.Dropout(TORCH_DROPOUT)
            self.head = nn.Linear(32, 2)

        def forward(self, x):
            output, _ = self.gru(x)
            return self.head(self.dropout(output[:, -1, :]))

    return SimpleGRUClassifier()


def make_ms_dlinear_tcn(input_dim, window_size, seed):
    torch = set_global_seed(seed)
    nn = torch.nn
    functional = torch.nn.functional

    class CausalConvBlock(nn.Module):
        def __init__(self, in_channels, out_channels, kernel_size, dropout):
            super().__init__()
            self.pad = kernel_size - 1
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size)
            self.norm = nn.BatchNorm1d(out_channels)
            self.dropout = nn.Dropout(dropout)
            self.proj = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

        def forward(self, x):
            residual = self.proj(x)
            padded = functional.pad(x, (self.pad, 0))
            out = self.conv(padded)
            out = self.dropout(torch.relu(self.norm(out)))
            return out + residual

    class MultiScaleDLinearTCNClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.tcn = nn.Sequential(
                CausalConvBlock(input_dim, TORCH_TCN_CHANNELS[0], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
                CausalConvBlock(TORCH_TCN_CHANNELS[0], TORCH_TCN_CHANNELS[1], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
            )
            self.scale_head = nn.Linear(input_dim * len(TORCH_MOVING_AVG_KERNELS), 16)
            self.head = nn.Linear(TORCH_TCN_CHANNELS[-1] + 16, 2)

        def moving_average_last(self, x, kernel):
            pad = kernel - 1
            padded = functional.pad(x.transpose(1, 2), (pad, 0), mode="replicate")
            avg = functional.avg_pool1d(padded, kernel_size=kernel, stride=1)
            return avg[:, :, -1]

        def forward(self, x):
            tcn_last = self.tcn(x.transpose(1, 2))[:, :, -1]
            scale_parts = [self.moving_average_last(x, kernel) for kernel in TORCH_MOVING_AVG_KERNELS]
            scale = torch.relu(self.scale_head(torch.cat(scale_parts, dim=1)))
            return self.head(torch.cat([tcn_last, scale], dim=1))

    return MultiScaleDLinearTCNClassifier()


def run_torch_shape_smoke(input_dim, window_size):
    torch = ensure_torch()
    for model_name, factory in (
        ("simple_gru", lambda: make_simple_gru(input_dim, 41)),
        ("ms_dlinear_tcn", lambda: make_ms_dlinear_tcn(input_dim, window_size, 41)),
    ):
        model = factory()
        x = torch.zeros((2, window_size, input_dim), dtype=torch.float32)
        y = torch.tensor([0, 1], dtype=torch.long)
        logits = model(x)
        if tuple(logits.shape) != (2, 2):
            raise ValueError(f"{model_name} shape smoke failed: logits shape {tuple(logits.shape)}")
        loss = torch.nn.CrossEntropyLoss()(logits, y)
        loss.backward()
    print("Deep adapter shape smoke passed.")


def fit_predict_torch_sequence(dataset, seed, model_name):
    torch = set_global_seed(seed)
    x_train, y_train, train_owner = subsample_rows_with_owner(
        dataset["x_train_seq"],
        dataset["y_train_seq"],
        dataset["train_owner_seq"],
        MAX_TRAIN_ROWS,
        seed=seed,
    )
    x_validation = dataset["x_validation_seq"]
    input_dim = x_train.shape[-1]
    window_size = x_train.shape[1]
    if model_name == "simple_gru":
        model = make_simple_gru(input_dim, seed)
    elif model_name == "ms_dlinear_tcn":
        model = make_ms_dlinear_tcn(input_dim, window_size, seed)
    else:
        raise ValueError(f"Unknown torch model: {model_name}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    train_x_tensor = torch.tensor(x_train, dtype=torch.float32)
    train_y_tensor = torch.tensor(y_train, dtype=torch.long)
    counts = np.bincount(y_train, minlength=2).astype(float)
    class_weights = counts.sum() / np.maximum(counts, 1.0)
    class_weights = class_weights / class_weights.mean()
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TORCH_LEARNING_RATE, weight_decay=TORCH_WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(train_x_tensor, train_y_tensor),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    start_fit = time.perf_counter()
    model.train()
    for _ in range(TORCH_EPOCHS):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    fit_seconds = time.perf_counter() - start_fit

    start_predict = time.perf_counter()
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(x_validation), TORCH_BATCH_SIZE):
            batch = torch.tensor(x_validation[start : start + TORCH_BATCH_SIZE], dtype=torch.float32, device=device)
            preds.append(model(batch).argmax(dim=1).cpu().numpy())
    predict_seconds = time.perf_counter() - start_predict
    return np.concatenate(preds), fit_seconds, predict_seconds, int(len(y_train)), "fixed_epochs_no_early_stopping"


def fit_predict_model(dataset, model_name, seed):
    if model_name == "logreg":
        return fit_predict_logreg(dataset, seed)
    if model_name == "lightgbm":
        return fit_predict_lightgbm(dataset, seed)
    if model_name in {"simple_gru", "ms_dlinear_tcn"}:
        return fit_predict_torch_sequence(dataset, seed, model_name)
    raise ValueError(f"Unknown model: {model_name}")


def concentration_from_per_ticker(per_ticker_rows):
    deltas = [row["per_ticker_delta_macro_f1_vs_dummy"] for row in per_ticker_rows]
    positive = [float(delta) for delta in deltas if pd.notna(delta) and delta > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    return positive_ticker_count, top_ticker_gain_share


def run_one_model_seed(stage, model_name, label_config, feature_set, window_size, seed):
    dataset = get_dataset(label_config, feature_set, window_size)
    prep_seconds = float(dataset["prep_seconds"])
    pred, fit_seconds, predict_seconds, train_n, fit_status = fit_predict_model(dataset, model_name, seed)
    pooled_metrics = evaluate_predictions(dataset["y_validation"], pred)
    pooled_dummy = dummy_metrics(dataset["y_train"], dataset["y_validation"], seed)
    per_ticker_rows = []
    for ticker in TICKERS:
        val_mask = dataset["validation_owner"] == ticker
        train_mask = dataset["train_owner"] == ticker
        if not val_mask.any():
            continue
        ticker_metrics = evaluate_predictions(dataset["y_validation"][val_mask], pred[val_mask])
        ticker_dummy = dummy_metrics(dataset["y_train"][train_mask], dataset["y_validation"][val_mask], seed)
        per_ticker_rows.append({
            "stage": stage,
            "model": model_name,
            "label_config": label_config,
            "horizon_k": dataset["horizon_k"],
            "threshold_bps": dataset["threshold_bps"],
            "feature_set": feature_set,
            "window_size": int(window_size),
            "seed": int(seed),
            "scope": RESULT_SCOPE,
            "macro_f1": ticker_metrics["macro_f1"],
            "balanced_accuracy": ticker_metrics["balanced_accuracy"],
            "accuracy": ticker_metrics["accuracy"],
            "dummy_macro_f1": ticker_dummy["dummy_macro_f1"],
            "dummy_balanced_accuracy": ticker_dummy["dummy_balanced_accuracy"],
            "delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "delta_balanced_accuracy_vs_dummy": (
                ticker_metrics["balanced_accuracy"] - ticker_dummy["dummy_balanced_accuracy"]
            ),
            "n": int(val_mask.sum()),
            "ticker_or_pooled": ticker,
            "prep_seconds": prep_seconds,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
            "per_ticker_delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "positive_ticker_count": np.nan,
            "top_ticker_gain_share": np.nan,
            "train_n": int(train_mask.sum()),
            "fit_status": fit_status,
        })
    positive_ticker_count, top_ticker_gain_share = concentration_from_per_ticker(per_ticker_rows)
    for row in per_ticker_rows:
        row["positive_ticker_count"] = positive_ticker_count
        row["top_ticker_gain_share"] = top_ticker_gain_share
    pooled_row = {
        "stage": stage,
        "model": model_name,
        "label_config": label_config,
        "horizon_k": dataset["horizon_k"],
        "threshold_bps": dataset["threshold_bps"],
        "feature_set": feature_set,
        "window_size": int(window_size),
        "seed": int(seed),
        "scope": RESULT_SCOPE,
        "macro_f1": pooled_metrics["macro_f1"],
        "balanced_accuracy": pooled_metrics["balanced_accuracy"],
        "accuracy": pooled_metrics["accuracy"],
        "dummy_macro_f1": pooled_dummy["dummy_macro_f1"],
        "dummy_balanced_accuracy": pooled_dummy["dummy_balanced_accuracy"],
        "delta_macro_f1_vs_dummy": pooled_metrics["macro_f1"] - pooled_dummy["dummy_macro_f1"],
        "delta_balanced_accuracy_vs_dummy": pooled_metrics["balanced_accuracy"] - pooled_dummy["dummy_balanced_accuracy"],
        "n": int(len(dataset["y_validation"])),
        "ticker_or_pooled": "pooled",
        "prep_seconds": prep_seconds,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
        "per_ticker_delta_macro_f1_vs_dummy": np.nan,
        "positive_ticker_count": positive_ticker_count,
        "top_ticker_gain_share": top_ticker_gain_share,
        "train_n": int(train_n),
        "fit_status": fit_status,
    }
    return pooled_row, per_ticker_rows


def run_stage_grid(stage, specs):
    pooled_rows = []
    per_ticker_rows = []
    for spec in specs:
        print(
            stage,
            spec["model"],
            spec["label_config"],
            spec["feature_set"],
            "window",
            spec["window_size"],
            "seed",
            spec["seed"],
        )
        pooled, per_ticker = run_one_model_seed(
            stage=stage,
            model_name=spec["model"],
            label_config=spec["label_config"],
            feature_set=spec["feature_set"],
            window_size=spec["window_size"],
            seed=spec["seed"],
        )
        pooled_rows.append(pooled)
        per_ticker_rows.extend(per_ticker)
    return pd.DataFrame(pooled_rows), pd.DataFrame(per_ticker_rows)


def summarize_pooled(pooled):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    keys = ["stage", "model", "label_config", "horizon_k", "threshold_bps", "feature_set", "window_size", "scope"]
    for key_values, group in pooled.groupby(keys, sort=False):
        record = dict(zip(keys, key_values))
        seed_count = int(group["seed"].nunique())
        macro_std = sample_std(group["macro_f1"])
        bal_std = sample_std(group["balanced_accuracy"])
        record.update({
            "seed_count": seed_count,
            "macro_f1_mean": float(group["macro_f1"].mean()),
            "macro_f1_std": macro_std,
            "macro_f1_lcb_95": float(
                group["macro_f1"].mean()
                - t_critical_one_sided_95(seed_count) * macro_std / math.sqrt(max(seed_count, 1))
            ),
            "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
            "balanced_accuracy_std": bal_std,
            "dummy_macro_f1_mean": float(group["dummy_macro_f1"].mean()),
            "dummy_balanced_accuracy_mean": float(group["dummy_balanced_accuracy"].mean()),
            "delta_macro_f1_vs_dummy_mean": float(group["delta_macro_f1_vs_dummy"].mean()),
            "delta_balanced_accuracy_vs_dummy_mean": float(group["delta_balanced_accuracy_vs_dummy"].mean()),
            "n_mean": float(group["n"].mean()),
            "positive_ticker_count": int(round(group["positive_ticker_count"].mean())),
            "top_ticker_gain_share": float(group["top_ticker_gain_share"].mean()),
            "prep_seconds_mean": float(group["prep_seconds"].mean()),
            "fit_seconds_mean": float(group["fit_seconds"].mean()),
            "predict_seconds_mean": float(group["predict_seconds"].mean()),
            "total_seconds_mean": float(group["total_seconds"].mean()),
        })
        record["basic_gate"] = bool(
            record["delta_macro_f1_vs_dummy_mean"] > 0
            and record["macro_f1_lcb_95"] > record["dummy_macro_f1_mean"]
        )
        record["lcb_eligible"] = bool(
            record["basic_gate"]
            and record["delta_balanced_accuracy_vs_dummy_mean"] > 0
            and record["top_ticker_gain_share"] < 0.50
            and record["positive_ticker_count"] >= 3
        )
        rows.append(record)
    return pd.DataFrame(rows)


def tuple_from_row(row, include_window):
    if row is None:
        return None
    if include_window:
        return {
            "label_config": row["label_config"],
            "feature_set": row["feature_set"],
            "window_size": int(row["window_size"]),
        }
    return {"label_config": row["label_config"], "feature_set": row["feature_set"]}


def select_candidates(summary, include_window):
    if summary.empty or not summary["basic_gate"].any():
        return {
            "stage0_result": "do_not_decide_config",
            "mean_candidate": None,
            "lcb_candidate": None,
            "candidate_count": 0,
        }
    basic = summary.loc[summary["basic_gate"]].sort_values("macro_f1_mean", ascending=False)
    lcb = summary.loc[summary["lcb_eligible"]].sort_values("macro_f1_lcb_95", ascending=False)
    mean_candidate = tuple_from_row(basic.iloc[0], include_window=include_window)
    lcb_candidate = tuple_from_row(lcb.iloc[0], include_window=include_window) if not lcb.empty else None
    unique_candidates = []
    for candidate in (mean_candidate, lcb_candidate):
        if candidate is not None and candidate not in unique_candidates:
            unique_candidates.append(candidate)
    return {
        "stage0_result": "candidate_config_selected",
        "mean_candidate": mean_candidate,
        "lcb_candidate": lcb_candidate,
        "candidate_count": len(unique_candidates),
        "candidates": unique_candidates,
    }


def write_outputs(pooled, per_ticker, summary, file_keys):
    pooled.to_csv(OUTPUT_FILES[file_keys[0]], index=False)
    per_ticker.to_csv(OUTPUT_FILES[file_keys[1]], index=False)
    if summary is not None:
        summary.to_csv(OUTPUT_FILES[file_keys[2]], index=False)
    print("wrote", [str(OUTPUT_FILES[key]) for key in file_keys])


## Diagnostic H0 Run Helpers

These helpers implement the H0 baseline read, Part 0 sanity gate, window sweep, LightGBM profiles, and fresh-seed confirmation.

In [ ]:
\
H0_STATE = {
    "baseline": None,
    "part0_passed": False,
}


def ensure_drive_backup_dir():
    if not BACKUP_TO_GOOGLE_DRIVE:
        return None
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    except ModuleNotFoundError as exc:
        message = "Google Drive backup requested, but google.colab is unavailable."
        if BACKUP_FAILURE_IS_FATAL:
            raise RuntimeError(message) from exc
        print("WARNING:", message)
        return None
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    return DRIVE_BACKUP_DIR


def try_mount_google_drive(reason):
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        return True
    except ModuleNotFoundError:
        print(f"Google Drive mount unavailable for {reason}; continuing without mount.")
        return False


def backup_h0_outputs(reason):
    if not BACKUP_TO_GOOGLE_DRIVE:
        return []
    backup_dir = ensure_drive_backup_dir()
    if backup_dir is None:
        return []
    copied = []
    try:
        for path in OUTPUT_FILES.values():
            if path.exists():
                target = backup_dir / path.name
                shutil.copy2(path, target)
                copied.append(str(target))
        manifest = {
            "diagnostic_name": DIAGNOSTIC_NAME,
            "reason": reason,
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "local_output_dir": str(OUTPUT_DIR),
            "drive_backup_dir": str(backup_dir),
            "copied_files": copied,
        }
        manifest_path = backup_dir / "diagnostic_h0_backup_manifest.json"
        with manifest_path.open("w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2)
        copied.append(str(manifest_path))
        print("Backed up H0 outputs to Drive:", backup_dir)
        return copied
    except Exception as exc:
        message = f"Google Drive backup failed after {reason}: {exc}"
        if BACKUP_FAILURE_IS_FATAL:
            raise RuntimeError(message) from exc
        print("WARNING:", message)
        return copied


def ensure_required_stage0_files():
    missing = [str(path) for path in STAGE0_FILES.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Diagnostic H0 requires completed Stage 0 outputs. Missing: "
            + ", ".join(missing)
        )


def missing_stage0_files():
    return [str(path) for path in STAGE0_FILES.values() if not path.exists()]


def find_stage0_artifact_dir():
    def valid_artifact_dir(path):
        return all((path / filename).exists() for filename in STAGE0_ARTIFACT_FILES.values())

    for path in STAGE0_ARTIFACT_DIR_CANDIDATES:
        if valid_artifact_dir(path):
            return path

    if any(str(path).startswith("/content/drive/") for path in STAGE0_ARTIFACT_DIR_CANDIDATES):
        try_mount_google_drive("Stage 0 artifact lookup")
        for path in STAGE0_ARTIFACT_DIR_CANDIDATES:
            if valid_artifact_dir(path):
                return path
    return None


def candidate_payload_contains_expected(payload):
    expected = {
        "label_config": H0_LABEL_CONFIG,
        "feature_set": H0_FEATURE_SET,
        "window_size": H0_BASELINE_WINDOW,
    }
    for candidate in payload.get("candidates", []):
        candidate_with_window = {
            "label_config": candidate.get("label_config"),
            "feature_set": candidate.get("feature_set"),
            "window_size": int(candidate.get("window_size")) if "window_size" in candidate else None,
        }
        if candidate_with_window == expected:
            return True
    return False


def read_stage0_artifact_candidate_payload(artifact_dir):
    path = artifact_dir / STAGE0_ARTIFACT_FILES["stage0_candidates"]
    payload = json.loads(path.read_text(encoding="utf-8"))
    blocks = payload if isinstance(payload, list) else [payload]
    for block in reversed(blocks):
        if candidate_payload_contains_expected(block):
            return block
    raise RuntimeError(
        "Stage 0 artifact decision blocks do not contain the intended H0 baseline candidate."
    )


def read_h0_baseline_from_artifacts():
    artifact_dir = find_stage0_artifact_dir()
    if artifact_dir is None:
        return None

    read_stage0_artifact_candidate_payload(artifact_dir)
    stage0b_path = artifact_dir / STAGE0_ARTIFACT_FILES["stage0b_summary"]
    stage0b = pd.read_csv(stage0b_path)
    if stage0b.empty:
        raise RuntimeError(f"Stage 0B artifact is empty: {stage0b_path}")

    summary_path = artifact_dir / STAGE0_ARTIFACT_FILES["stage0a2_summary"]
    summary = pd.read_csv(summary_path)
    summary_mask = (
        summary["model"].eq(H0_BASELINE_MODEL)
        & summary["label_config"].eq(H0_LABEL_CONFIG)
        & summary["feature_set"].eq(H0_FEATURE_SET)
        & summary["window_size"].astype(int).eq(H0_BASELINE_WINDOW)
    )
    if int(summary_mask.sum()) != 1:
        raise RuntimeError(
            "Expected exactly one artifact Stage 0A2 summary row for the H0 baseline, "
            f"found {int(summary_mask.sum())} in {summary_path}."
        )
    row = summary.loc[summary_mask].iloc[0].to_dict()
    seed_count = int(row.get("seed_count", 0))
    if seed_count != len(MODEL_SEEDS):
        raise RuntimeError(
            f"Artifact baseline seed_count mismatch: expected {len(MODEL_SEEDS)}, found {seed_count}."
        )
    baseline = {
        "macro_f1_mean": float(row["macro_f1_mean"]),
        "summary_row": row,
        "source": f"stage0_extracted_artifact:{summary_path}",
        "artifact_dir": str(artifact_dir),
    }
    H0_STATE["baseline"] = baseline
    print("Using extracted Stage 0 artifact baseline:", summary_path)
    print("H0 baseline macro_f1_mean:", baseline["macro_f1_mean"])
    return baseline


def read_stage0_candidates():
    ensure_required_stage0_files()
    with STAGE0_FILES["stage0_candidates"].open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("stage0_result") == "do_not_decide_config":
        raise RuntimeError("Stage 0 reported do_not_decide_config; Diagnostic H0 must not run.")
    candidates = payload.get("candidates", [])
    expected = {
        "label_config": H0_LABEL_CONFIG,
        "feature_set": H0_FEATURE_SET,
        "window_size": H0_BASELINE_WINDOW,
    }
    if expected not in candidates:
        raise RuntimeError(
            "The intended H0 baseline is not an official Stage 0A2 candidate. "
            f"Expected {expected}; found {candidates}. Revise the H0 protocol before running."
        )
    return payload


def read_h0_baseline():
    missing = missing_stage0_files()
    if missing:
        artifact_baseline = read_h0_baseline_from_artifacts()
        if artifact_baseline is not None:
            return artifact_baseline
        if not ALLOW_STANDALONE_H0_BASELINE:
            raise FileNotFoundError(
                "Diagnostic H0 requires completed Stage 0 outputs. Missing: "
                + ", ".join(missing)
            )
        baseline = {
            "macro_f1_mean": None,
            "summary_row": {},
            "source": "part0_standalone_pending",
            "missing_stage0_files": missing,
        }
        H0_STATE["baseline"] = baseline
        print("WARNING: Stage 0 output files are missing; using standalone H0 baseline mode.")
        print("Missing Stage 0 files:", missing)
        print("Part 0 will define h0_baseline_macro_f1 for this diagnostic run.")
        return baseline

    read_stage0_candidates()
    summary = pd.read_csv(STAGE0_FILES["stage0a2_summary"])
    pooled = pd.read_csv(STAGE0_FILES["stage0a2_pooled"])

    summary_mask = (
        summary["model"].eq(H0_BASELINE_MODEL)
        & summary["label_config"].eq(H0_LABEL_CONFIG)
        & summary["feature_set"].eq(H0_FEATURE_SET)
        & summary["window_size"].astype(int).eq(H0_BASELINE_WINDOW)
    )
    if int(summary_mask.sum()) != 1:
        raise RuntimeError(
            "Expected exactly one Stage 0A2 summary row for the H0 baseline, "
            f"found {int(summary_mask.sum())}."
        )
    row = summary.loc[summary_mask].iloc[0].to_dict()

    pooled_mask = (
        pooled["model"].eq(H0_BASELINE_MODEL)
        & pooled["label_config"].eq(H0_LABEL_CONFIG)
        & pooled["feature_set"].eq(H0_FEATURE_SET)
        & pooled["window_size"].astype(int).eq(H0_BASELINE_WINDOW)
    )
    observed_seeds = tuple(sorted(int(seed) for seed in pooled.loc[pooled_mask, "seed"].unique()))
    if observed_seeds != MODEL_SEEDS:
        raise RuntimeError(
            f"Baseline pooled seeds mismatch: expected {MODEL_SEEDS}, found {observed_seeds}."
        )

    baseline = {
        "macro_f1_mean": float(row["macro_f1_mean"]),
        "summary_row": row,
        "source": str(STAGE0_FILES["stage0a2_summary"]),
    }
    H0_STATE["baseline"] = baseline
    return baseline


def require_h0_baseline():
    if H0_STATE["baseline"] is None:
        return read_h0_baseline()
    return H0_STATE["baseline"]


def require_part0_passed():
    if not H0_STATE["part0_passed"]:
        raise RuntimeError("Run Diagnostic H0 Part 0 successfully before this part.")
    return require_h0_baseline()


def fit_predict_logreg_h0(dataset, seed, params):
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    max_iter = 2000
    model = LogisticRegression(
        solver=params.get("solver", "liblinear"),
        penalty=params.get("penalty", "l2"),
        C=float(params.get("C", 1.0)),
        max_iter=max_iter,
        class_weight="balanced",
        random_state=seed,
    )
    start_fit = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    max_iter_reached = bool((model.n_iter_ >= max_iter).any())
    fit_status = "converged" if not convergence_warnings and not max_iter_reached else "convergence_warning"
    return pred, fit_seconds, predict_seconds, int(len(y_train)), fit_status


def fit_predict_lightgbm_h0(dataset, seed, params):
    lgb = ensure_lightgbm()
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    lgbm_params = dict(params)
    lgbm_params.setdefault("class_weight", "balanced")
    model = lgb.LGBMClassifier(
        **lgbm_params,
        random_state=seed,
        verbosity=-1,
    )
    start_fit = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    return pred, fit_seconds, predict_seconds, int(len(y_train)), "not_applicable"


def fit_predict_h0_model(dataset, spec):
    model_name = spec["model"]
    seed = int(spec["seed"])
    params = spec.get("params", {})
    if model_name == "logreg":
        return fit_predict_logreg_h0(dataset, seed, params)
    if model_name == "lightgbm":
        return fit_predict_lightgbm_h0(dataset, seed, params)
    raise ValueError(f"Diagnostic H0 supports only tabular models, got {model_name!r}.")


def concentration_from_per_ticker_h0(per_ticker_rows):
    deltas = [row["per_ticker_delta_macro_f1_vs_dummy"] for row in per_ticker_rows]
    positive = [float(delta) for delta in deltas if pd.notna(delta) and delta > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    return positive_ticker_count, top_ticker_gain_share


def run_one_h0_spec(spec):
    dataset = get_dataset(spec["label_config"], spec["feature_set"], spec["window_size"])
    prep_seconds = float(dataset["prep_seconds"])
    pred, fit_seconds, predict_seconds, train_n, fit_status = fit_predict_h0_model(dataset, spec)
    pooled_metrics = evaluate_predictions(dataset["y_validation"], pred)
    pooled_dummy = dummy_metrics(dataset["y_train"], dataset["y_validation"], int(spec["seed"]))
    params_json = json.dumps(spec.get("params", {}), sort_keys=True)

    base_fields = {
        "diagnostic_name": DIAGNOSTIC_NAME,
        "stage": "Diagnostic H0",
        "part": spec["part"],
        "round": spec.get("round", "none"),
        "model": spec["model"],
        "profile": spec["profile"],
        "label_config": spec["label_config"],
        "horizon_k": dataset["horizon_k"],
        "threshold_bps": dataset["threshold_bps"],
        "feature_set": spec["feature_set"],
        "window_size": int(spec["window_size"]),
        "seed": int(spec["seed"]),
        "scope": RESULT_SCOPE,
        "params_json": params_json,
    }

    per_ticker_rows = []
    for ticker in TICKERS:
        val_mask = dataset["validation_owner"] == ticker
        train_mask = dataset["train_owner"] == ticker
        if not val_mask.any():
            continue
        ticker_metrics = evaluate_predictions(dataset["y_validation"][val_mask], pred[val_mask])
        ticker_dummy = dummy_metrics(dataset["y_train"][train_mask], dataset["y_validation"][val_mask], int(spec["seed"]))
        per_ticker_rows.append({
            **base_fields,
            "macro_f1": ticker_metrics["macro_f1"],
            "balanced_accuracy": ticker_metrics["balanced_accuracy"],
            "accuracy": ticker_metrics["accuracy"],
            "dummy_macro_f1": ticker_dummy["dummy_macro_f1"],
            "dummy_balanced_accuracy": ticker_dummy["dummy_balanced_accuracy"],
            "delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "delta_balanced_accuracy_vs_dummy": (
                ticker_metrics["balanced_accuracy"] - ticker_dummy["dummy_balanced_accuracy"]
            ),
            "n": int(val_mask.sum()),
            "ticker_or_pooled": ticker,
            "prep_seconds": prep_seconds,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
            "per_ticker_delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "positive_ticker_count": np.nan,
            "top_ticker_gain_share": np.nan,
            "train_n": int(train_mask.sum()),
            "fit_status": fit_status,
        })

    positive_ticker_count, top_ticker_gain_share = concentration_from_per_ticker_h0(per_ticker_rows)
    for row in per_ticker_rows:
        row["positive_ticker_count"] = positive_ticker_count
        row["top_ticker_gain_share"] = top_ticker_gain_share

    pooled_row = {
        **base_fields,
        "macro_f1": pooled_metrics["macro_f1"],
        "balanced_accuracy": pooled_metrics["balanced_accuracy"],
        "accuracy": pooled_metrics["accuracy"],
        "dummy_macro_f1": pooled_dummy["dummy_macro_f1"],
        "dummy_balanced_accuracy": pooled_dummy["dummy_balanced_accuracy"],
        "delta_macro_f1_vs_dummy": pooled_metrics["macro_f1"] - pooled_dummy["dummy_macro_f1"],
        "delta_balanced_accuracy_vs_dummy": pooled_metrics["balanced_accuracy"] - pooled_dummy["dummy_balanced_accuracy"],
        "n": int(len(dataset["y_validation"])),
        "ticker_or_pooled": "pooled",
        "prep_seconds": prep_seconds,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
        "per_ticker_delta_macro_f1_vs_dummy": np.nan,
        "positive_ticker_count": positive_ticker_count,
        "top_ticker_gain_share": top_ticker_gain_share,
        "train_n": int(train_n),
        "fit_status": fit_status,
    }
    return pooled_row, per_ticker_rows


def run_h0_grid(specs):
    pooled_rows = []
    per_ticker_rows = []
    for idx, spec in enumerate(specs, start=1):
        print(
            f"{idx}/{len(specs)}",
            spec["part"],
            spec["model"],
            spec["profile"],
            spec["label_config"],
            spec["feature_set"],
            "window",
            spec["window_size"],
            "seed",
            spec["seed"],
        )
        pooled, per_ticker = run_one_h0_spec(spec)
        pooled_rows.append(pooled)
        per_ticker_rows.extend(per_ticker)
    return pd.DataFrame(pooled_rows), pd.DataFrame(per_ticker_rows)


def summarize_h0_pooled(pooled, baseline_macro_f1):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    keys = [
        "diagnostic_name",
        "part",
        "round",
        "model",
        "profile",
        "label_config",
        "horizon_k",
        "threshold_bps",
        "feature_set",
        "window_size",
        "scope",
    ]
    for key_values, group in pooled.groupby(keys, sort=False):
        record = dict(zip(keys, key_values))
        seed_count = int(group["seed"].nunique())
        macro_std = sample_std(group["macro_f1"])
        bal_std = sample_std(group["balanced_accuracy"])
        macro_mean = float(group["macro_f1"].mean())
        delta_vs_base = macro_mean - float(baseline_macro_f1)
        record.update({
            "seed_count": seed_count,
            "baseline_macro_f1": float(baseline_macro_f1),
            "baseline_source": str((H0_STATE.get("baseline") or {}).get("source", "unknown")),
            "macro_f1_mean": macro_mean,
            "macro_f1_std": macro_std,
            "macro_f1_lcb_95": float(
                macro_mean - t_critical_one_sided_95(seed_count) * macro_std / math.sqrt(max(seed_count, 1))
            ),
            "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
            "balanced_accuracy_std": bal_std,
            "dummy_macro_f1_mean": float(group["dummy_macro_f1"].mean()),
            "dummy_balanced_accuracy_mean": float(group["dummy_balanced_accuracy"].mean()),
            "delta_macro_f1_vs_dummy_mean": float(group["delta_macro_f1_vs_dummy"].mean()),
            "delta_balanced_accuracy_vs_dummy_mean": float(group["delta_balanced_accuracy_vs_dummy"].mean()),
            "delta_macro_f1_vs_base": delta_vs_base,
            "n_mean": float(group["n"].mean()),
            "positive_ticker_count": int(round(group["positive_ticker_count"].mean())),
            "top_ticker_gain_share": float(group["top_ticker_gain_share"].mean()),
            "prep_seconds_mean": float(group["prep_seconds"].mean()),
            "fit_seconds_mean": float(group["fit_seconds"].mean()),
            "predict_seconds_mean": float(group["predict_seconds"].mean()),
            "total_seconds_mean": float(group["total_seconds"].mean()),
            "fit_status_values": ",".join(sorted(str(value) for value in group["fit_status"].dropna().unique())),
        })
        record["preliminary_gate"] = bool(
            record["delta_macro_f1_vs_base"] >= H0_WEAK_DELTA
            and record["delta_macro_f1_vs_dummy_mean"] > 0
            and record["positive_ticker_count"] >= H0_MIN_POSITIVE_TICKERS
            and record["top_ticker_gain_share"] <= H0_MAX_TOP_TICKER_GAIN_SHARE
        )
        if record["delta_macro_f1_vs_base"] < H0_WEAK_DELTA:
            record["interpretation"] = "indistinguishable_from_base"
        elif record["delta_macro_f1_vs_base"] < H0_STRONG_DELTA:
            record["interpretation"] = "weak_diagnostic_preference_note_only"
        else:
            record["interpretation"] = "strong_diagnostic_signal_requires_new_branch"
        if record["part"] == "part3_confirmation":
            record["confirmation_status"] = (
                "confirmed"
                if record["delta_macro_f1_vs_base"] >= H0_WEAK_DELTA
                else "lottery_not_confirmed"
            )
        else:
            record["confirmation_status"] = "not_selected_for_confirmation"
        rows.append(record)
    return pd.DataFrame(rows)


def existing_part_frames(kind):
    paths = {
        "part0": OUTPUT_FILES["part0"],
        "part1": OUTPUT_FILES["part1"],
        "part2": OUTPUT_FILES["part2"],
        "part3": OUTPUT_FILES["part3"],
    }
    frames = []
    for path in paths.values():
        if path.exists():
            frames.append(pd.read_csv(path))
    return frames


def write_h0_outputs(part_key, pooled, per_ticker, baseline_macro_f1):
    pooled.to_csv(OUTPUT_FILES[part_key], index=False)
    per_ticker_frames = []
    existing_per_ticker = OUTPUT_FILES["per_ticker"]
    if existing_per_ticker.exists():
        prior = pd.read_csv(existing_per_ticker)
        prior = prior.loc[~prior["part"].eq(pooled["part"].iloc[0])]
        per_ticker_frames.append(prior)
    per_ticker_frames.append(per_ticker)
    pd.concat(per_ticker_frames, ignore_index=True).to_csv(OUTPUT_FILES["per_ticker"], index=False)

    pooled_frames = existing_part_frames("pooled")
    combined = pd.concat(pooled_frames, ignore_index=True) if pooled_frames else pooled
    summary = summarize_h0_pooled(combined, baseline_macro_f1)
    summary.to_csv(OUTPUT_FILES["summary"], index=False)
    print("wrote", str(OUTPUT_FILES[part_key]), str(OUTPUT_FILES["summary"]), str(OUTPUT_FILES["per_ticker"]))
    backup_h0_outputs(f"completed_{part_key}")
    return summary


def make_lgbm_spec(part, round_name, profile, window_size, seed):
    return {
        "part": part,
        "round": round_name,
        "model": "lightgbm",
        "profile": profile,
        "label_config": H0_LABEL_CONFIG,
        "feature_set": H0_FEATURE_SET,
        "window_size": int(window_size),
        "seed": int(seed),
        "params": LGBM_PROFILES[profile].copy(),
    }


def make_logreg_spec(part, round_name, profile, window_size, seed):
    return {
        "part": part,
        "round": round_name,
        "model": "logreg",
        "profile": profile,
        "label_config": H0_LABEL_CONFIG,
        "feature_set": H0_FEATURE_SET,
        "window_size": int(window_size),
        "seed": int(seed),
        "params": LOGREG_PROFILES[profile].copy(),
    }


def build_part0_specs():
    return [
        make_lgbm_spec("part0_sanity", "part0", H0_BASELINE_PROFILE, H0_BASELINE_WINDOW, seed)
        for seed in MODEL_SEEDS
    ]


def build_part1_specs():
    specs = [
        make_lgbm_spec("part1_window_sweep", "round1", H0_BASELINE_PROFILE, window_size, seed)
        for window_size in H0_PART1_LGBM_WINDOWS
        for seed in MODEL_SEEDS
    ]
    specs.extend(
        make_logreg_spec("part1_window_sweep", "round1", "logreg_l2_c1", window_size, seed)
        for window_size in H0_PART1_LOGREG_WINDOWS
        for seed in MODEL_SEEDS
    )
    return specs


def build_round2_specs():
    return [
        make_lgbm_spec("part1_window_sweep", "round2", H0_BASELINE_PROFILE, window_size, seed)
        for window_size in H0_ROUND2_WINDOWS
        for seed in MODEL_SEEDS
    ]


def build_part2_specs():
    return [
        make_lgbm_spec("part2_lgbm_profiles", "part2", profile, window_size, seed)
        for profile in LGBM_PROFILES
        for window_size in H0_PART2_WINDOWS
        for seed in MODEL_SEEDS
    ]


def round2_is_triggered(summary):
    if summary.empty:
        return False
    mask = (
        summary["part"].eq("part1_window_sweep")
        & summary["round"].eq("round1")
        & summary["model"].eq("lightgbm")
        & summary["window_size"].astype(int).isin(H0_ROUND2_TRIGGER_WINDOWS)
        & (summary["delta_macro_f1_vs_base"] >= H0_WEAK_DELTA)
        & (summary["positive_ticker_count"] >= H0_MIN_POSITIVE_TICKERS)
        & (summary["top_ticker_gain_share"] <= H0_MAX_TOP_TICKER_GAIN_SHARE)
    )
    return bool(mask.any())


def spec_from_summary_row(row, seed):
    if row["model"] == "lightgbm":
        return make_lgbm_spec("part3_confirmation", "fresh_seed", row["profile"], int(row["window_size"]), seed)
    if row["model"] == "logreg":
        return make_logreg_spec("part3_confirmation", "fresh_seed", row["profile"], int(row["window_size"]), seed)
    raise ValueError(f"Unsupported confirmation model: {row['model']}")


def build_confirmation_specs():
    if not OUTPUT_FILES["summary"].exists():
        raise FileNotFoundError("No H0 summary found. Run Part 1 and/or Part 2 first.")
    summary = pd.read_csv(OUTPUT_FILES["summary"])
    eligible = summary.loc[
        summary["preliminary_gate"].astype(bool)
        & ~summary["part"].eq("part3_confirmation")
    ].sort_values("delta_macro_f1_vs_base", ascending=False)
    top = eligible.head(3)
    if top.empty:
        print("No cells cleared preliminary gates; Part 3 has no specs.")
        return []
    specs = []
    for _, row in top.iterrows():
        for seed in H0_FRESH_SEEDS:
            specs.append(spec_from_summary_row(row, seed))
    display(top)
    return specs


def assert_logreg_warning_rate_ok(pooled):
    mask = pooled["model"].eq("logreg")
    if not mask.any():
        return
    warning_rate = float(pooled.loc[mask, "fit_status"].eq("convergence_warning").mean())
    if warning_rate > H0_LOGREG_WARNING_RATE_ABORT:
        raise RuntimeError(
            f"LogReg convergence warning rate {warning_rate:.3f} exceeds "
            f"{H0_LOGREG_WARNING_RATE_ABORT:.3f}; stop and inspect."
        )


## Part 0 - Sanity Check

Part 0 rereads the official Stage 0A2 baseline dynamically and reruns the exact
baseline cell in this new notebook. It must reproduce the official
`macro_f1_mean` within `1e-4`, otherwise H0 aborts.


In [ ]:
\
if RUN_H0_PART0:
    baseline = read_h0_baseline()
    print("H0 baseline macro_f1_mean before Part 0:", baseline["macro_f1_mean"])
    part0_pooled, part0_per_ticker = run_h0_grid(build_part0_specs())
    provisional_baseline = (
        float(part0_pooled["macro_f1"].mean())
        if baseline["macro_f1_mean"] is None
        else float(baseline["macro_f1_mean"])
    )
    part0_summary = summarize_h0_pooled(part0_pooled, provisional_baseline)
    display(part0_summary)
    part0_value = float(part0_summary["macro_f1_mean"].iloc[0])
    if baseline["macro_f1_mean"] is None:
        baseline = {
            **baseline,
            "macro_f1_mean": part0_value,
            "source": "part0_standalone",
        }
        H0_STATE["baseline"] = baseline
        print("Standalone H0 baseline macro_f1_mean:", part0_value)
        print("This run cannot claim reproduction of official Stage 0A2 output.")
    else:
        diff = abs(part0_value - baseline["macro_f1_mean"])
        if diff > H0_BASELINE_TOLERANCE:
            part0_pooled.to_csv(OUTPUT_FILES["part0"], index=False)
            backup_h0_outputs("part0_failed_baseline_reproduction")
            raise RuntimeError(
                f"Part 0 failed baseline reproduction: diff={diff:.8f}, "
                f"tolerance={H0_BASELINE_TOLERANCE}."
            )
    H0_STATE["part0_passed"] = True
    h0_summary = write_h0_outputs("part0", part0_pooled, part0_per_ticker, baseline["macro_f1_mean"])
else:
    print("RUN_H0_PART0 is False; Part 0 sanity check not run.")


## Part 1 - Window Sweep

Part 1 runs sparse window anchors. Round 2 dense sweep is conditional and only
runs when the pre-committed trigger is met and `RUN_H0_ROUND2_IF_TRIGGERED` is
also set to True.


In [ ]:
\
if RUN_H0_PART1:
    baseline = require_part0_passed()
    part1_specs = build_part1_specs()
    part1_pooled, part1_per_ticker = run_h0_grid(part1_specs)
    assert_logreg_warning_rate_ok(part1_pooled)
    part1_summary = summarize_h0_pooled(part1_pooled, baseline["macro_f1_mean"])
    display(part1_summary.sort_values("delta_macro_f1_vs_base", ascending=False))

    if round2_is_triggered(part1_summary):
        print("Round 2 trigger met.")
        if RUN_H0_ROUND2_IF_TRIGGERED:
            round2_pooled, round2_per_ticker = run_h0_grid(build_round2_specs())
            part1_pooled = pd.concat([part1_pooled, round2_pooled], ignore_index=True)
            part1_per_ticker = pd.concat([part1_per_ticker, round2_per_ticker], ignore_index=True)
            part1_summary = summarize_h0_pooled(part1_pooled, baseline["macro_f1_mean"])
            display(part1_summary.sort_values("delta_macro_f1_vs_base", ascending=False))
        else:
            print("RUN_H0_ROUND2_IF_TRIGGERED is False; Round 2 not run.")
    else:
        print("Round 2 trigger not met.")

    h0_summary = write_h0_outputs("part1", part1_pooled, part1_per_ticker, baseline["macro_f1_mean"])
else:
    print("RUN_H0_PART1 is False; Part 1 window sweep not run.")


## Part 2 - LightGBM Profile Sweep

Part 2 runs the neutral LightGBM profiles from the H0 protocol over windows
20, 24, and 32.


In [ ]:
\
if RUN_H0_PART2:
    baseline = require_part0_passed()
    part2_pooled, part2_per_ticker = run_h0_grid(build_part2_specs())
    part2_summary = summarize_h0_pooled(part2_pooled, baseline["macro_f1_mean"])
    display(part2_summary.sort_values("delta_macro_f1_vs_base", ascending=False))
    h0_summary = write_h0_outputs("part2", part2_pooled, part2_per_ticker, baseline["macro_f1_mean"])
else:
    print("RUN_H0_PART2 is False; Part 2 LightGBM profile sweep not run.")


## Part 3 - Fresh-Seed Confirmation

Part 3 selects the top 3 preliminary-gate cells from existing H0 summary output
and reruns only those cells with fresh seeds.


In [ ]:
\
if RUN_H0_PART3:
    baseline = require_part0_passed()
    confirmation_specs = build_confirmation_specs()
    if confirmation_specs:
        part3_pooled, part3_per_ticker = run_h0_grid(confirmation_specs)
        part3_summary = summarize_h0_pooled(part3_pooled, baseline["macro_f1_mean"])
        part3_summary["confirmation_status"] = np.where(
            part3_summary["delta_macro_f1_vs_base"] >= H0_WEAK_DELTA,
            "confirmed",
            "lottery_not_confirmed",
        )
        display(part3_summary.sort_values("delta_macro_f1_vs_base", ascending=False))
        h0_summary = write_h0_outputs("part3", part3_pooled, part3_per_ticker, baseline["macro_f1_mean"])
    else:
        print("No confirmation specs generated; Part 3 skipped.")
else:
    print("RUN_H0_PART3 is False; Part 3 confirmation not run.")


## Interpretation Boundary

Diagnostic H0 is `validation_only_diagnostic`.

Allowed wording:

```text
Diagnostic H0 found a validation-only diagnostic preference for [cell], but the
result is post-Stage 0 and cannot replace the official Stage 0 selection without
a new pre-registered branch.
```

Forbidden wording:

```text
Window 24 is the best configuration.
The official Stage 0 winner should be replaced.
This result is ready for holdout/test.
The thesis model is selected.
```

Cells with `delta_macro_f1_vs_base < 0.005` are treated as indistinguishable
from the official baseline. Cells between `0.005` and `0.010` are weak
diagnostic notes only. Cells at or above `0.010` require a new pre-registered
branch before use.
